In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=4dee66d633a6a2d53071102d6be1aef6d2353f04126e25f15314320d185994f4
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.


In [3]:
# Generate random bit using quantum measurement
def quantum_random_bit():

    qc = QuantumCircuit(1, 1)

    qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    return int(list(counts.keys())[0])

In [4]:
# Generate a sequence of random bits
def generate_random_bits(n):

    bits = []

    for i in range(n):
        bits.append(quantum_random_bit())

    return bits

In [5]:
# Encode a bit using either the standard or diagonal basis
def prepare_qubit(bit, basis):

    qc = QuantumCircuit(1, 1)

    # Encode bit value
    if bit == 1:
        qc.x(0)

    # Convert to diagonal basis
    if basis == 1:
        qc.h(0)

    return qc

In [6]:
# Measure a qubit using the chosen basis
def measure_qubit(qc, basis):

    # Convert diagonal basis back to standard basis
    if basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    return int(list(counts.keys())[0])

In [7]:
n = 100

# =========================
# Alice's part
# =========================

## Alice generates random bits and bases
alice_bits = generate_random_bits(n)
alice_bases = generate_random_bits(n)

qubits = []

for i in range(n):

    qubit = prepare_qubit(
        alice_bits[i],
        alice_bases[i]
    )

    qubits.append(qubit)


In [8]:
# =========================
# Bob's part
# =========================

## Bob randomly choose bases and measure received qubits
bob_bases = generate_random_bits(n)

bob_bits = []

for i in range(n):

    measured_bit = measure_qubit(
        qubits[i],
        bob_bases[i]
    )

    bob_bits.append(measured_bit)

In [9]:
# ==================
# Public discussion
# ==================
matching_positions = []

for i in range(n):

    if alice_bases[i] == bob_bases[i]:
        matching_positions.append(i)

alice_key = []
bob_key = []

# Keep positions where Alice and Bob use the same basis
for i in matching_positions:

    alice_key.append(alice_bits[i])
    bob_key.append(bob_bits[i])

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

print("Bob bases:  ", bob_bases)
print("Bob bits:   ", bob_bits)

print("Matching positions:", matching_positions)

print("Alice key:", alice_key)
print("Bob key:  ", bob_key)

print("Keys match:", alice_key == bob_key)

Alice bits:  [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0]
Alice bases: [1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1]
Bob bases:   [1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1]
Bob bits:    [1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1,

In [10]:
# The results show that Alice and Bob only keep the bits where their bases match.
# At these matching positions, Bob measures the same bit values as Alice.
# Therefore, without an attacker, both participants successfully generate the same shared key.